# broadcasting-rules — ex6: pairwise distance matrix as a heatmap

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `broadcasting-rules`. Running the final beacon cell reports progress against the `Numpy: Vectorization and broadcasting` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Numpy: Vectorization and broadcasting` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`broadcasting-rules`** (exercise 6). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "broadcasting-rules"
DD_SUBTOPIC = "Numpy: Vectorization and broadcasting"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Broadcasting — quick refresher

**The rule** (NumPy & PyTorch agree):
1. Right-align both shapes; left-pad the shorter with 1s.
2. For each pair of aligned axes: equal → keep; one is 1 → use the other; otherwise → incompatible.

**The dangerous case.** When a shape *almost* matches you can get an unintended broadcast that runs silently and produces wrong values. Always shape-check (`print(x.shape, y.shape, (x*y).shape)`) when wiring up a new pipeline.

### Exercise 6 — pairwise distance matrix as a heatmap

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Compute a pairwise Euclidean distance matrix via broadcast + reduce, then visualize it as a heatmap.
> Keywords: broadcast, reduce, heatmap, matplotlib, axis-insertion
> ```

**KCs targeted:** `insert-axis-for-broadcast`, `broadcast-then-reduce`, `pairwise-via-broadcast`

Given two point sets `A` of shape `(N, D)` and `B` of shape `(M, D)`, implement `ex6_pairwise_distances(A, B)` to return a tensor `D_mat` of shape `(N, M)` where `D_mat[i, j]` is the Euclidean distance between `A[i]` and `B[j]`.

**Do it without any Python loops.** The trick is to insert axes so the subtraction broadcasts:
- `A[:, None, :]` has shape `(N, 1, D)`
- `B[None, :, :]` has shape `(1, M, D)`
- their difference is `(N, M, D)`; square-and-sum over the last axis, then `sqrt`.

After the function works, the test cell will render the resulting `(N, M)` distance matrix as a matplotlib heatmap so you can *see* the broadcast result — a diagonal-dark stripe when `A == B`, a smooth gradient otherwise.

**Shape-trace it in your head before coding:** `(N, 1, D) - (1, M, D) → (N, M, D) → sum(-1) → (N, M) → sqrt → (N, M)`.

In [ ]:
def ex6_pairwise_distances(A: Tensor, B: Tensor) -> Tensor:
    """Return (N, M) tensor of Euclidean distances.

    A: (N, D), B: (M, D). No Python loops — pure broadcast.
    """
    raise NotImplementedError()


def _test_ex6():
    # Small deterministic case we can verify by hand
    A = t.tensor([[0.0, 0.0], [1.0, 0.0], [0.0, 1.0]])
    B = t.tensor([[0.0, 0.0], [1.0, 1.0]])
    D_mat = ex6_pairwise_distances(A, B)
    assert D_mat.shape == (3, 2), f'expected (3, 2), got {tuple(D_mat.shape)}'
    # Hand-checked distances
    expected = t.tensor([
        [0.0, 2.0 ** 0.5],
        [1.0, 1.0],
        [1.0, 1.0],
    ])
    assert t.allclose(D_mat, expected, atol=1e-6), f'value mismatch:\n{D_mat}'

    # Self-distance must have a zero diagonal
    P = t.randn(20, 4)
    DD = ex6_pairwise_distances(P, P)
    assert DD.shape == (20, 20)
    assert t.allclose(DD.diagonal(), t.zeros(20), atol=1e-5), 'self-distance diagonal must be 0'
    assert t.allclose(DD, DD.T, atol=1e-5), 'self-distance must be symmetric'

    # Visualize — heatmap of distances between two random point clouds
    import matplotlib.pyplot as plt
    AA = t.randn(30, 2)
    BB = t.randn(25, 2) + 0.5
    DM = ex6_pairwise_distances(AA, BB).numpy()
    fig, ax = plt.subplots(figsize=(5, 6))
    im = ax.imshow(DM, aspect='auto', cmap='viridis')
    ax.set_xlabel('B index'); ax.set_ylabel('A index')
    ax.set_title(f'pairwise distances  ({DM.shape[0]} × {DM.shape[1]})')
    plt.colorbar(im, ax=ax, label='distance')
    plt.tight_layout(); plt.show()
    _dd_passed.add('ex6')
    print("ex6 ✓")

_test_ex6()

<details><summary>Solution</summary>

```python
def ex6_pairwise_distances(A: Tensor, B: Tensor) -> Tensor:
    diff = A[:, None, :] - B[None, :, :]   # (N, M, D)
    sq = (diff ** 2).sum(dim=-1)            # (N, M)
    return sq.sqrt()
```

**Why broadcast beats a double loop.** The naive `for i in range(N): for j in range(M): ...` version is `O(N·M)` Python overhead. The broadcast version is one big vectorized op the BLAS / GPU kernel can fuse. For `N = M = 1000, D = 64`, the loop version is ~4 orders of magnitude slower.

**Memory cost.** Intermediate `diff` is `(N, M, D)` — for large N, M, D this can be huge. The production-grade alternative is `‖a-b‖² = ‖a‖² + ‖b‖² − 2 a·bᵀ` (Gram-matrix trick), which never materializes the `(N, M, D)` block.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex6'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex6',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()